# Enum — Nível Avançado

**Objetivo:** usar Enums como peça de arquitetura — máquinas de estado, estratégias, enums gerados em runtime,
contratos abertos para APIs externas, exaustividade verificada, persistência segura e performance.

Roteiro:

1. Máquina de estados declarada no Enum
2. Strategy pattern com `member()` (3.11+)
3. API funcional: Enums gerados em runtime (config, banco, planilha)
4. `FlagBoundary`: o que fazer com bits desconhecidos
5. Enum "aberto" para APIs externas (`UNKNOWN` sem quebrar o pipeline)
6. Exaustividade com `assert_never`
7. Persistência: `value`, nunca nome ou posição
8. Enums em uniões discriminadas (Pydantic)
9. Internals: `EnumType`, `_value2member_map_`, `nonmember`, `enum.property`
10. Performance
11. Checklist

In [1]:
import json, timeit, sqlite3, logging
from enum import Enum, StrEnum, IntEnum, Flag, IntFlag, auto, member, nonmember, FlagBoundary, STRICT, CONFORM, EJECT, KEEP
from typing import assert_never, Literal, Annotated, Union

## 1. Máquina de estados declarada no Enum

O ciclo de vida de um pedido, de um ticket ou de um job é uma **máquina de estados**. Declarar as transições permitidas
**junto** do Enum transforma a regra de negócio em dado verificável (e desenhável) em vez de `if`s espalhados.

In [2]:
class InvalidTransition(Exception):
    pass

class OrderState(StrEnum):
    CART = auto()
    PENDING_PAYMENT = auto()
    PAID = auto()
    SHIPPED = auto()
    DELIVERED = auto()
    CANCELLED = auto()
    REFUNDED = auto()

    # nonmember: atributo de classe que NÃO vira membro
    _TRANSITIONS = nonmember({
        "cart": {"pending_payment", "cancelled"},
        "pending_payment": {"paid", "cancelled"},
        "paid": {"shipped", "refunded"},
        "shipped": {"delivered"},
        "delivered": {"refunded"},
        "cancelled": set(),
        "refunded": set(),
    })

    @property
    def allowed(self) -> set["OrderState"]:
        return {OrderState(v) for v in self._TRANSITIONS[self.value]}

    @property
    def is_terminal(self) -> bool:
        return not self.allowed

    def to(self, target: "OrderState") -> "OrderState":
        if target not in self.allowed:
            raise InvalidTransition(f"{self} -> {target} não permitido (permitidos: {sorted(self.allowed)})")
        return target

state = OrderState.CART
for nxt in (OrderState.PENDING_PAYMENT, OrderState.PAID, OrderState.SHIPPED, OrderState.DELIVERED):
    state = state.to(nxt)
    print(f"-> {state}")

try:
    OrderState.SHIPPED.to(OrderState.CANCELLED)
except InvalidTransition as e:
    print(f"InvalidTransition: {e}")
print(f"terminais: {[s for s in OrderState if s.is_terminal]}")

-> pending_payment
-> paid
-> shipped
-> delivered
InvalidTransition: shipped -> cancelled não permitido (permitidos: [<OrderState.DELIVERED: 'delivered'>])
terminais: [<OrderState.CANCELLED: 'cancelled'>, <OrderState.REFUNDED: 'refunded'>]


Como a regra é **dado**, dá para validá-la e gerar documentação automaticamente — por exemplo, um diagrama Mermaid:

In [3]:
# sanidade: todo estado alcançável a partir de CART?
reach, frontier = {OrderState.CART}, [OrderState.CART]
while frontier:
    for n in frontier.pop().allowed - reach:
        reach.add(n); frontier.append(n)
print(f'inalcançáveis: {set(OrderState) - reach or "nenhum"}')

print("stateDiagram-v2")
print("    [*] --> cart")
for s in OrderState:
    for t in sorted(s.allowed):
        print(f"    {s} --> {t}")
    if s.is_terminal:
        print(f"    {s} --> [*]")

inalcançáveis: nenhum
stateDiagram-v2
    [*] --> cart
    cart --> cancelled
    cart --> pending_payment
    pending_payment --> cancelled
    pending_payment --> paid
    paid --> refunded
    paid --> shipped
    shipped --> delivered
    delivered --> refunded
    cancelled --> [*]
    refunded --> [*]


## 2. Strategy pattern com `member()`

Cada membro pode carregar **comportamento**. Funções definidas no corpo do Enum normalmente viram métodos (não membros);
`enum.member(...)` (3.11+) força a função a virar o **valor** do membro. Resultado: o vocabulário e a implementação ficam juntos,
e adicionar uma estratégia nova é adicionar um membro.

In [4]:
from decimal import Decimal, ROUND_HALF_UP

def _q(v: Decimal) -> Decimal:
    return v.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

class ShippingRule(Enum):
    FLAT = member(lambda weight_kg, subtotal: Decimal("19.90"))
    BY_WEIGHT = member(lambda weight_kg, subtotal: _q(Decimal("5") + Decimal("4.5") * Decimal(str(weight_kg))))
    FREE_ABOVE_200 = member(lambda weight_kg, subtotal: Decimal("0.00") if subtotal >= 200 else Decimal("24.90"))

    def __call__(self, weight_kg: float, subtotal: Decimal) -> Decimal:
        return self.value(weight_kg, subtotal)

for rule in ShippingRule:
    print(f"{rule.name:<15} 3kg/R$150 -> {rule(3, Decimal('150')):>6}   3kg/R$250 -> {rule(3, Decimal('250')):>6}")

# a estratégia vem da configuração, como texto
print(f'config \'BY_WEIGHT\': {ShippingRule["BY_WEIGHT"](10, Decimal("80"))}')

FLAT            3kg/R$150 ->  19.90   3kg/R$250 ->  19.90
BY_WEIGHT       3kg/R$150 ->  18.50   3kg/R$250 ->  18.50
FREE_ABOVE_200  3kg/R$150 ->  24.90   3kg/R$250 ->   0.00
config 'BY_WEIGHT': 50.00


## 3. API funcional: Enums gerados em runtime

`Enum("Nome", ...)` cria um Enum a partir de uma lista/dict. Útil quando o vocabulário vem de **fora do código** mas deve ser
**congelado no startup**: categorias de um arquivo de config, códigos de uma tabela de referência, colunas de uma planilha.
Você ganha validação e `match`/autocomplete parcial sem hardcode.

In [5]:
import tomllib

CONFIG = tomllib.loads("""
[catalog]
categories = ["eletronicos", "livros", "casa", "esporte"]

[sla_hours]
low = 72
normal = 24
urgent = 4
""")

Category = StrEnum("Category", {c.upper(): c for c in CONFIG["catalog"]["categories"]})
SlaTier = IntEnum("SlaTier", {k.upper(): v for k, v in CONFIG["sla_hours"].items()})

print(f"list(Category): {list(Category)}")
print(f'Category("livros"): {Category("livros")}')
print(f"é str?: {isinstance(Category.LIVROS, str)}")
print(f"sorted(SlaTier): {sorted(SlaTier)}")
print(f"mais rápido: {min(SlaTier).name}")

# valida dados contra o vocabulário carregado
products = [{"sku": "1", "category": "livros"}, {"sku": "2", "category": "brinquedos"}]
for p in products:
    status = "ok" if p["category"] in Category else f"categoria inválida ({p['category']})"
    print(f"sku {p['sku']}: {status}")

list(Category): [<Category.ELETRONICOS: 'eletronicos'>, <Category.LIVROS: 'livros'>, <Category.CASA: 'casa'>, <Category.ESPORTE: 'esporte'>]
Category("livros"): livros
é str?: True
sorted(SlaTier): [<SlaTier.URGENT: 4>, <SlaTier.NORMAL: 24>, <SlaTier.LOW: 72>]
mais rápido: URGENT
sku 1: ok
sku 2: categoria inválida (brinquedos)


> Trade-off: o type checker não conhece membros criados em runtime. Use API funcional quando a flexibilidade compensa;
> para vocabulários que o código precisa referenciar por nome (`Category.LIVROS` num `if`), prefira a classe declarada.

## 4. `FlagBoundary`: bits desconhecidos

O que acontece quando chega um valor de `Flag` com bits que você não definiu (ex.: um sistema novo enviou uma permissão nova)?
O parâmetro `boundary` decide (3.11+):

| boundary | comportamento | padrão de |
|---|---|---|
| `STRICT` | erro | `Flag` |
| `CONFORM` | descarta os bits desconhecidos | — |
| `EJECT` | devolve um `int` comum (sai do Enum) | — |
| `KEEP` | mantém os bits extras dentro do flag | `IntFlag` |

In [6]:
raw_from_other_system = 0b1_0011          # bits 1, 2 conhecidos; bit 16 desconhecido

for boundary in (STRICT, CONFORM, EJECT, KEEP):
    Perm = Flag("Perm", {"READ": 1, "WRITE": 2, "DELETE": 4}, boundary=boundary)
    try:
        v = Perm(raw_from_other_system)
        print(f"{boundary.name:<8} -> {v!r}  ({type(v).__name__})")
    except ValueError as e:
        print(f"{boundary.name:<8} -> ValueError: {str(e).splitlines()[0]}")

STRICT   -> ValueError: <flag 'Perm'> invalid value 19
CONFORM  -> <Perm.READ|WRITE: 3>  (Perm)
EJECT    -> 19  (int)
KEEP     -> <Perm.READ|WRITE|16: 19>  (Perm)


Escolha consciente: **STRICT** quando o valor vem de você mesmo (bit desconhecido = bug); **CONFORM** para entrada externa em que
ignorar o desconhecido é seguro; **KEEP** quando você precisa repassar os bits intactos para outro sistema.

## 5. Enum "aberto" para APIs externas

Você **consome** a API de um provedor de pagamentos que hoje devolve `status` em `{approved, declined, pending}`.
Amanhã ele adiciona `"under_review"`. Com um Enum fechado, seu worker quebra em produção. Com um Enum "aberto",
o valor novo vira `UNKNOWN`, é **registrado** (log/métrica) e o processamento segue por um caminho seguro.

In [7]:
log = logging.getLogger("provider")
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)

class ProviderStatus(StrEnum):
    APPROVED = auto()
    DECLINED = auto()
    PENDING = auto()
    UNKNOWN = auto()

    @classmethod
    def _missing_(cls, value):
        log.warning("status desconhecido do provedor: %r (tratando como UNKNOWN)", value)
        return cls.UNKNOWN

def handle_payment(payload: dict) -> str:
    status = ProviderStatus(payload["status"])
    match status:
        case ProviderStatus.APPROVED:
            return "liberar pedido"
        case ProviderStatus.DECLINED:
            return "notificar cliente"
        case ProviderStatus.PENDING | ProviderStatus.UNKNOWN:
            return "manter em espera e revisar"     # caminho seguro

for s in ["approved", "declined", "under_review"]:
    print(f"{s:<13} -> {handle_payment({'status': s})}")

WARNING provider: status desconhecido do provedor: 'under_review' (tratando como UNKNOWN)


approved      -> liberar pedido
declined      -> notificar cliente
under_review  -> manter em espera e revisar


> Para APIs que você **expõe**, mantenha o Enum **fechado**: o cliente precisa saber tudo o que pode receber, e
> adicionar membro é uma mudança de contrato (versione ou comunique). Abertura é para o que você **consome**.

## 6. Exaustividade com `assert_never`

`case _: raise` pega o problema em **runtime**. Com `typing.assert_never` no `case _`, o **type checker** (mypy/pyright)
acusa *antes de rodar* se algum membro ficou sem tratamento — por exemplo, quando alguém adiciona `REFUNDED` ao Enum
e esquece de atualizar o `match`.

In [8]:
class Channel(Enum):
    EMAIL = "email"
    SMS = "sms"
    PUSH = "push"

def send(channel: Channel, msg: str) -> str:
    match channel:
        case Channel.EMAIL:
            return f"email: {msg}"
        case Channel.SMS:
            return f"sms: {msg[:160]}"
        case Channel.PUSH:
            return f"push: {msg[:80]}"
        case _:
            assert_never(channel)      # mypy/pyright: erro se faltar algum membro acima

print(f'[send(c, "Seu pedido saiu para entrega") for c in Channel]: {[send(c, "Seu pedido saiu para entrega") for c in Channel]}')

[send(c, "Seu pedido saiu para entrega") for c in Channel]: ['email: Seu pedido saiu para entrega', 'sms: Seu pedido saiu para entrega', 'push: Seu pedido saiu para entrega']


Se você remover o `case Channel.PUSH`, o pyright reporta algo como:
`Argument of type "Literal[Channel.PUSH]" cannot be assigned to parameter of type "Never"`.
Em runtime, o `assert_never` também levanta `AssertionError` caso um valor inesperado chegue.

## 7. Persistência: `value`, nunca nome ou posição

| o que gravar | renomear membro | reordenar membros | adicionar no meio | legível no banco |
|---|---|---|---|---|
| `value` explícito (`"paid"`) | ✅ seguro | ✅ | ✅ | ✅ |
| `name` (`"PAID"`) | ❌ quebra | ✅ | ✅ | ✅ |
| `auto()` inteiro | ✅ | ❌ quebra | ❌ quebra | ❌ |
| índice `list(E).index(m)` | ✅ | ❌ | ❌ | ❌ |

O `value` é o **contrato** com o banco. Trate mudanças nele como migração de dados.

In [9]:
class TicketStatus(StrEnum):
    OPEN = "open"
    IN_PROGRESS = "in_progress"
    RESOLVED = "resolved"

# sqlite3: adaptador grava o value; conversão na leitura
sqlite3.register_adapter(TicketStatus, lambda s: s.value)
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE tickets (id INTEGER PRIMARY KEY, status TEXT NOT NULL "
           "CHECK (status IN (%s)))" % ",".join(f"'{s.value}'" for s in TicketStatus))
db.executemany("INSERT INTO tickets (status) VALUES (?)",
               [(TicketStatus.OPEN,), (TicketStatus.RESOLVED,), (TicketStatus.OPEN,)])

rows = [(i, TicketStatus(s)) for i, s in db.execute("SELECT id, status FROM tickets")]
print(f"rows: {rows}")
print(f'db.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall(): {db.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall()}')
try:
    db.execute("INSERT INTO tickets (status) VALUES ('closed')")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError (CHECK gerado do Enum): {e}")

rows: [(1, <TicketStatus.OPEN: 'open'>), (2, <TicketStatus.RESOLVED: 'resolved'>), (3, <TicketStatus.OPEN: 'open'>)]
db.execute("SELECT status, COUNT(*) FROM tickets GROUP BY status").fetchall(): [('open', 2), ('resolved', 1)]
IntegrityError (CHECK gerado do Enum): CHECK constraint failed: status IN ('open','in_progress','resolved')


Em ORMs a armadilha se repete: o `Enum` do SQLAlchemy grava o **name** por padrão. Comparando o padrão com a correção
(`values_callable`, que faz gravar o **value**):

In [10]:
from sqlalchemy import create_engine, text, Enum as SAEnum
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, Session

class Base(DeclarativeBase):
    pass

class TicketDefault(Base):                       # armadilha: grava "IN_PROGRESS"
    __tablename__ = "tickets_default"
    id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[TicketStatus] = mapped_column(SAEnum(TicketStatus))

class TicketFixed(Base):                         # correção: grava "in_progress"
    __tablename__ = "tickets_fixed"
    id: Mapped[int] = mapped_column(primary_key=True)
    status: Mapped[TicketStatus] = mapped_column(
        SAEnum(TicketStatus, values_callable=lambda e: [m.value for m in e], native_enum=False))

engine = create_engine("sqlite://")
Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([TicketDefault(status=TicketStatus.IN_PROGRESS), TicketFixed(status=TicketStatus.IN_PROGRESS)])
    session.commit()
    raw_default = session.execute(text("SELECT status FROM tickets_default")).scalar_one()
    raw_fixed = session.execute(text("SELECT status FROM tickets_fixed")).scalar_one()
    loaded = session.query(TicketFixed).one().status

print(f"gravado no banco (padrão): {raw_default}")
print(f"gravado no banco (values_callable): {raw_fixed}")
print(f"lido de volta como Enum: {loaded!r}")

gravado no banco (padrão): IN_PROGRESS
gravado no banco (values_callable): in_progress
lido de volta como Enum: <TicketStatus.IN_PROGRESS: 'in_progress'>


> Django: `models.TextChoices` já grava o value.

## 8. Enums em uniões discriminadas (Pydantic)

O tipo do evento costuma ser um `StrEnum` — e cada modelo da união fixa **um** membro com `Literal`. O Pydantic usa isso como
discriminador: roteamento O(1) e JSON Schema com `const` por variante.

In [11]:
from pydantic import BaseModel, Field, TypeAdapter, ValidationError

class EventType(StrEnum):
    USER_SIGNED_UP = "user.signed_up"
    ORDER_PLACED = "order.placed"

class UserSignedUp(BaseModel):
    type: Literal[EventType.USER_SIGNED_UP]
    user_id: int

class OrderPlaced(BaseModel):
    type: Literal[EventType.ORDER_PLACED]
    order_id: int
    total: float

Event = Annotated[Union[UserSignedUp, OrderPlaced], Field(discriminator="type")]
EventsAdapter = TypeAdapter(list[Event])

events = EventsAdapter.validate_json('[{"type": "order.placed", "order_id": 9, "total": 99.9}, {"type": "user.signed_up", "user_id": 3}]')
print(f"[(type(e).__name__, e.type) for e in events]: {[(type(e).__name__, e.type) for e in events]}")
try:
    EventsAdapter.validate_python([{"type": "user.deleted", "user_id": 1}])
except ValidationError as e:
    print(f'e.errors(include_url=False)[0]["type"]: {e.errors(include_url=False)[0]["type"]}')

[(type(e).__name__, e.type) for e in events]: [('OrderPlaced', <EventType.ORDER_PLACED: 'order.placed'>), ('UserSignedUp', <EventType.USER_SIGNED_UP: 'user.signed_up'>)]
e.errors(include_url=False)[0]["type"]: union_tag_invalid


## 9. Internals úteis

* `EnumType` (metaclasse) mantém `_member_map_` (nome → membro) e `_value2member_map_` (valor → membro).
* `__members__` é um *mappingproxy* somente-leitura, **incluindo aliases**.
* `nonmember(x)` guarda `x` como atributo de classe comum (usado na seção 1).
* `enum.property` permite ter um membro e uma propriedade com o mesmo nome sem conflito (é o que `.name`/`.value` usam).
* Herança: só de Enums **sem membros** — o jeito de compartilhar métodos entre vários Enums (*mixin*).

In [12]:
print(f"type(OrderState).__name__: {type(OrderState).__name__}")
print(f"list(OrderState._value2member_map_.items())[:3]: {list(OrderState._value2member_map_.items())[:3]}")

class LabeledMixin(Enum):
    """Enum base SEM membros: só comportamento compartilhado."""
    @property
    def label(self) -> str:
        return self.name.replace("_", " ").title()

class Department(LabeledMixin):
    HUMAN_RESOURCES = 1
    SALES_OPS = 2

class Shift(LabeledMixin):
    NIGHT_SHIFT = "n"

print(f"Department.SALES_OPS.label: {Department.SALES_OPS.label}")
print(f"Shift.NIGHT_SHIFT.label: {Shift.NIGHT_SHIFT.label}")
try:
    class MoreDepartments(Department):
        LEGAL = 3
except TypeError as e:
    print(f"TypeError: {e}")

type(OrderState).__name__: EnumType
list(OrderState._value2member_map_.items())[:3]: [('cart', <OrderState.CART: 'cart'>), ('pending_payment', <OrderState.PENDING_PAYMENT: 'pending_payment'>), ('paid', <OrderState.PAID: 'paid'>)]
Department.SALES_OPS.label: Sales Ops
Shift.NIGHT_SHIFT.label: Night Shift
TypeError: <enum 'MoreDepartments'> cannot extend <enum 'Department'>


## 10. Performance

Conversões via `Enum(valor)` passam pela metaclasse; em loops muito quentes, um `dict` pré-montado é mais rápido.
Acesso a membro e comparação por identidade são praticamente gratuitos.

In [13]:
values = ["open", "in_progress", "resolved"] * 1000
LOOKUP = dict(TicketStatus._value2member_map_)        # cópia estável do índice interno
members = [TicketStatus(v) for v in values]
N = 200

cases = {
    "TicketStatus(v)": lambda: [TicketStatus(v) for v in values],
    "LOOKUP[v] (dict pré-montado)": lambda: [LOOKUP[v] for v in values],
    "s is TicketStatus.OPEN": lambda: [s is TicketStatus.OPEN for s in members],
    "s == 'open' (StrEnum vs str)": lambda: [s == "open" for s in members],
}
for name, fn in cases.items():
    t = min(timeit.repeat(fn, number=N, repeat=3)) / N / 3000 * 1e9
    print(f"{name:<32} {t:6.1f} ns/item")

TicketStatus(v)                   145.9 ns/item
LOOKUP[v] (dict pré-montado)       19.2 ns/item
s is TicketStatus.OPEN             23.2 ns/item
s == 'open' (StrEnum vs str)       21.7 ns/item


Na prática: só otimize com `dict` se o profiler apontar a conversão como gargalo (ex.: parse de milhões de linhas de log/CSV).

## 11. Checklist

| ✔ | prática |
|---|---|
| ☐ | Valores persistidos/expostos são **explícitos** e tratados como contrato (nada de `auto()` inteiro persistido) |
| ☐ | `StrEnum` quando o valor vive em JSON/HTTP; `Enum` puro quando quer impedir mistura com strings |
| ☐ | `@unique` / `@verify(UNIQUE)` em vocabulários de domínio; em Flags, `auto()` + combinações com nomes + teste dos aliases esperados (`@verify(NAMED_FLAGS)` só pega bits sem nome) |
| ☐ | Texto externo (SQL, cache, URL, logs estruturados) sempre com `.value` |
| ☐ | Entrada humana/legada normalizada em `_missing_`, num lugar só |
| ☐ | APIs **consumidas**: enum aberto com `UNKNOWN` + log; APIs **expostas**: enum fechado |
| ☐ | `match` com `assert_never` para exaustividade verificada pelo type checker |
| ☐ | Regras do vocabulário (transições, labels, limites) como métodos/dados do próprio Enum |
| ☐ | `CHECK`/tipo enum no banco gerado a partir do Enum (fonte única da verdade) |

### Exercícios
1. Adicione `RETURNED` à máquina de estados (a partir de `DELIVERED`) e gere o diagrama de novo.
2. Transforme `ShippingRule` para ler a tabela de preços de um TOML (API funcional + `member`).
3. Crie um teste que falhe se alguém adicionar membro a `Channel` sem atualizar `send` (dica: itere `Channel` e chame `send`).
4. Faça um `Flag` de escopos OAuth (`read:user`, `write:repo`...) com `_missing_` que aceite a string `"read:user write:repo"`.

➡️ Casos de uso com LLM: [`../AI_use_cases/README.md`](../AI_use_cases/README.md)